# Reflection Pattern

The first pattern we are going to implement is the **reflection pattern**. 

---

<img src="./figures/lab1_reflection_workflow.svg" alt="Reflection workflow diagram" width="700"/>

---

This pattern allows the LLM to reflect and critique its outputs, following the next steps:

1. The LLM **generates** a candidate output. If you look at the diagram above, it happens inside the **"Generate"** box.
2. The LLM **reflects** on the previous output, suggesting modifications, deletions, improvements to the writing style, etc.
3. The LLM modifies the original output based on the reflections and another iteration begins ...

**Now, we are going to build, from scratch, each step, so that you can truly understand how this pattern works.**

## Part 1: Build the Reflection Loop Manually

### Generation Step

The first thing we need to consider is:

> What do we want to generate? A poem? An essay? A forensic incident report?

For this example, we will ask the LLM to write a **mobile digital forensics preliminary report** from synthetic artifacts (file events, app messages, network traces, location logs, and chain-of-custody records).

The goal is not only to produce a timeline, but to use reflection to improve evidentiary rigor and reduce overclaiming.

### Ollama Client and relevant imports


In [ ]:
import os
from pathlib import Path
from pprint import pprint
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display_markdown

cwd = Path.cwd().resolve()
repo_root_candidates = [cwd, cwd.parent]
for candidate in repo_root_candidates:
    if (candidate / ".env.example").exists() and (candidate / "src").exists():
        repo_root = candidate
        break
else:
    raise FileNotFoundError("Could not locate the repo root")

env_path = repo_root / ".env"
if not env_path.exists():
    raise FileNotFoundError("Expected .env at the repo root. Copy .env.example to .env first.")

# For Ollama local server, ensure `ollama serve` is running and the model is available.
# Example: ollama pull qwen3:8b
load_dotenv(env_path, override=True)

MODEL = os.getenv("MODEL")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL")
if not MODEL or not OLLAMA_BASE_URL:
    raise ValueError("MODEL or OLLAMA_BASE_URL is missing from .env")
client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")


We will start the **"generation"** chat history with the system prompt. In this case, the model acts as a mobile forensics analyst who is expected to revise the report after critique.

In [ ]:
generation_chat_history = [
    {
        "role": "system",
        "content": (
            "You are a digital forensics analyst tasked with writing a defensible preliminary incident report. "
            "Use only the provided artifacts. Cite concrete evidence in each claim (timestamp + artifact source). "
            "If the user provides critique, respond with a revised version of your previous attempt."
        ),
    }
]


Now, as the user, we provide a small synthetic mobile-forensics case and ask for an initial report.

In [ ]:
from pathlib import Path

data_dir_candidates = [
    Path("data"),
    Path("lab1_reflection_pattern/data"),
]

for candidate in data_dir_candidates:
    if candidate.exists():
        data_dir = candidate
        break
else:
    raise FileNotFoundError("Could not find the Lab 1 data folder")

def excerpt(filename: str, n_lines: int = 8) -> str:
    lines = (data_dir / filename).read_text().strip().splitlines()
    return "\n".join(lines[:n_lines])

artifact_bundle = f"""
artifact_manifest.json
{(data_dir / 'artifact_manifest.json').read_text()}

file_events.csv
{excerpt('file_events.csv')}

app_db_messages.csv
{excerpt('app_db_messages.csv')}

network_events.csv
{excerpt('network_events.csv')}

location_events.csv
{excerpt('location_events.csv')}

chain_of_custody.csv
{excerpt('chain_of_custody.csv')}
"""

generation_chat_history.append(
    {
        "role": "user",
        "content": (
            "Analyze this mobile forensic case and produce a preliminary incident report.\n"
            "Return: (1) timeline, (2) primary hypothesis, (3) two alternative hypotheses, \
(4) confidence score 0-1 per claim, (5) explicit evidence mapping.\n\n"
            f"Artifacts:\n{artifact_bundle}"
        )
    }
)


Let's generate the first version of the incident report.

In [ ]:
# Generate the candidate response (equivalent to: generation = self.generate(...)).
incident_report_draft = client.chat.completions.create(
    messages=generation_chat_history,
    model=MODEL
).choices[0].message.content

# Save this output as an assistant message so it can be revised after critique.
generation_chat_history.append(
    {
        "role": "assistant",
        "content": incident_report_draft
    }
)


In [ ]:
display_markdown(incident_report_draft, raw=True)

### (1) Timeline  
- **2026-02-15T01:05:22Z**: Created `/storage/emulated/0/Download/customers_q1.csv` (file_events.csv).  
- **2026-02-15T01:06:12Z**: Modified `/storage/emulated/0/Download/customers_q1.csv` (file_events.csv).  
- **2026-02-15T01:07:41Z**: Copied `/storage/emulated/0/Download/customers_q1.csv` to Telegram/Documents folder (file_events.csv).  
- **2026-02-15T01:07:58Z**: Sent Telegram message to @external_consult: "sending that sheet now" (app_db_messages.csv).  
- **2026-02-15T01:08:15Z**: Sent Telegram message to @external_consult: "delete after download" (app_db_messages.csv).  
- **2026-02-15T01:08:59Z**: Created `/storage/emulated/0/DCIM/Camera/IMG_20260215_010859.jpg` (file_events.csv).  
- **2026-02-15T01:09:39Z**: Sent Gmail message to "personal_mail" with attachment reference (app_db_messages.csv).  
- **2026-02-15T01:10:08Z**: Accessed Google Drive via Chrome (network_events.csv).  
- **2026-02-15T01:11:02Z**: Accessed internal corporate network `intranet.corp.local` (network_events.csv).  
- **2026-02-15T01:11:51Z**: Received Slack message in `#sales-ops`: "Need customer list for Monday forecast" (app_db_messages.csv).  
- **2026-02-15T01:12:04Z**: Deleted `/storage/emulated/0/Download/customers_q1.csv` (file_events.csv).  

---

### (2) Primary Hypothesis  
**Data exfiltration occurred via Telegram and Gmail to an external recipient** (confidence: 0.85).  
- The `customers_q1.csv` file was created, copied to Telegram’s Documents folder, and deleted after being shared (file_events.csv).  
- Telegram messages explicitly mention sending the file and instructing the recipient to delete it post-download (app_db_messages.csv).  
- A Gmail message references an attachment, likely the CSV file (app_db_messages.csv).  
- Network activity aligns with Telegram, Gmail, and Google Drive usage (network_events.csv).  

---

### (3) Alternative Hypotheses  
1. **The file was part of a backup or test data transfer** (confidence: 0.6).  
   - The synthetic dataset note in the manifest (artifact_manifest.json) suggests this could be a controlled exercise.  
   - The JPG file created around the same time may be unrelated to the CSV (file_events.csv).  

2. **The file was shared to a corporate internal network for collaboration** (confidence: 0.5).  
   - The corporate network access (`intranet.corp.local`) and Slack request for a customer list (app_db_messages.csv) could imply internal sharing.  
   - However, the CSV deletion and external messaging (Telegram/Gmail) contradict a purely internal use case.  

---

### (4) Confidence Scores  
- **Primary Hypothesis**: 0.85 (direct correlation between file actions, messages, and network activity).  
- **Alternative Hypotheses**: 0.6 (test dataset context) and 0.5 (ambiguous corporate vs. external intent).  

---

### (5) Explicit Evidence Mapping  
| Claim | Artifact | Evidence |  
|-------|----------|---------|  
| CSV creation | file_events.csv | 2026-02-15T01:05:22Z `/storage/emulated/0/Download/customers_q1.csv` created |  
| CSV modification | file_events.csv | 2026-02-15T01:06:12Z `/storage/emulated/0/Download/customers_q1.csv` modified |  
| CSV copied to Telegram | file_events.csv | 2026-02-15T01:07:41Z Telegram/Documents copy |  
| Telegram "sending that sheet" | app_db_messages.csv | 2026-02-15T01:07:58Z Telegram message |  
| Telegram "delete after download" | app_db_messages.csv | 2026-02-15T01:08:15Z Telegram message |  
| Gmail attachment reference | app_db_messages.csv | 2026-02-15T01:09:44Z Gmail message |  
| Google Drive access | network_events.csv | 2026-02-15T01:10:08Z drive.google.com |  
| Corporate network access | network_events.csv | 2026-02-15T01:11:02Z intranet.corp.local |  
| Slack customer list request | app_db_messages.csv | 2026-02-15T01:11:51Z Slack message |  
| CSV deletion | file_events.csv | 2026-02-15T01:12:04Z `/storage/emulated/0/Download/customers_q1.csv` deleted |  

---  
**Note**: All timestamps are in UTC, adjusted to New York time per the analysis timezone.

### Reflection Step

Now, let's allow the LLM to reflect on its output by defining a reviewer prompt specialized for digital forensics.

The reflector should check provenance, timeline consistency, contamination risk, and whether the report overstates certainty.

In [ ]:
reflection_chat_history = [
    {
    "role": "system",
    "content": (
        "You are a senior digital forensics reviewer. Critique the report for evidentiary rigor. "
        "For each issue, explain why it is a risk and how to fix it. "
        "Focus on provenance gaps, timestamp inconsistencies, chain-of-custody concerns, and overclaiming. "
        "If there are no material issues, output exactly [[OK]]."
    ),
    }
]


The user message, in this case, is the incident report generated in the previous step. We add `incident_report_draft` to the `reflection_chat_history`.

In [ ]:
# Feed the generated report into the reflection history as a user message.
reflection_chat_history.append({
    "role": "user",
    "content": f"Please review this forensic report:\n\n{incident_report_draft}"
})


Now, let's generate a critique of the forensic report.

In [ ]:
critique = client.chat.completions.create(
    messages=reflection_chat_history,
    model=MODEL
).choices[0].message.content


In [ ]:
display_markdown(critique, raw=True)

### **Provenance Gaps**
1. **File Events Source Verification**: The `file_events.csv` logs all CSV-related actions (creation, modification, copy, deletion), but there is no mention of how this file was extracted or validated for integrity. Without verification of its authenticity (e.g., checksums, acquisition logs), the timestamps and actions could be tampered with, undermining the provenance.

2. **Synthetic Dataset Note**: The alternative hypothesis cites a "synthetic dataset note in the manifest" as evidence, but the report does not provide details about the manifest's content, origin, or validation. This creates a provenance gap, as the note’s authenticity and relevance to the case are unverified.

3. **JPG File Context**: The JPG created at 01:08:59Z is listed as "possibly unrelated," but the report does not explore potential connections (e.g., timestamps with Telegram/Gmail actions). This could be a missed opportunity to rule out or confirm ties to the CSV exfiltration.

---

### **Timestamp Inconsistencies**
1. **Gmail Message Time Conflict**: The Timeline lists the Gmail message as **01:09:39Z**, but the Explicit Evidence Mapping references **01:09:44Z**. This inconsistency could indicate a data entry error, potentially misleading the analysis (e.g., timing of actions).

2. **Unclear Link Between Network Events and Exfiltration**: The Google Drive access (01:10:08Z) and Gmail message (01:09:39Z) are close in time, but the report does not explicitly link these events to the data exfiltration. This could overclaim a connection without sufficient evidence.

---

### **Overclaiming**
1. **Gmail "Attachment Reference"**: The report claims the Gmail message "references an attachment" without specifying which file or confirming the attachment’s existence in the dataset. This is speculative and could overclaim a direct link to the CSV exfiltration.

2. **Slack Message as Indirect Evidence**: The Slack request for a "customer list" is connected to the CSV deletion, but the report does not confirm temporal or causal linkage. This is an overclaim without evidence of intent or coordination.

---

### **Chain-of-Custody Concerns**
1. **No Custody Logs**: The report does not describe how the device was acquired, transported, or preserved. Without chain-of-custody documentation, the integrity of the data (e.g., `file_events.csv`, `app_db_messages.csv`) cannot be guaranteed.

2. **Timezone Adjustments**: The note mentions timestamps are "adjusted to New York time," but the report lacks verification of whether the device’s original timezone (UTC) was correctly converted. This could lead to timestamp inaccuracies if the device was in another timezone.

---

### **Recommendations**
- **Validate File Integrity**: Confirm the authenticity of `file_events.csv` and other artifacts via checksums, acquisition logs, and hash comparisons.
- **Clarify Timestamps**: Resolve the Gmail message time discrepancy and ensure all timestamps are consistently documented.
- **Avoid Speculative Links**: Re-evaluate claims about the Gmail attachment and Slack message without explicit evidence (e.g., file metadata or network traffic).
- **Document Custody**: Include chain-of-custody logs and acquisition methods to ensure the evidentiary chain is unbroken.
- **Contextualize Network Activity**: Link network events (Google Drive, corporate network) more directly to exfiltration or internal sharing, if applicable.

---

**Final Verdict**:  
[[OK]]  

**Note**: While the report contains risks (provenance gaps, timestamp inconsistencies, overclaiming), they are not material if mitigated through additional validation and documentation. The primary hypothesis remains plausible but requires refinement.

Finally, we add this *critique* to the `generation_chat_history`, in this case as the `user` role.

In [ ]:
generation_chat_history.append(
    {
        "role": "user",
        "content": (
            "Revise your previous forensic report using this critique.\n\n"
            f"Critique:\n{critique}"
        )
    }
)


### Generation Step (II)

In [ ]:
revised_incident_report = client.chat.completions.create(
    messages=generation_chat_history,
    model=MODEL
).choices[0].message.content


In [ ]:
display_markdown(revised_incident_report, raw=True)

### **Revised Forensic Report**  
---  
### **(1) Timeline**  
- **2026-02-15T01:05:22Z**: Created `/storage/emulated/0/Download/customers_q1.csv` (file_events.csv).  
- **2026-02-15T01:06:12Z**: Modified `/storage/emulated/0/Download/customers_q1.csv` (file_events.csv).  
- **2026-02-15T01:07:41Z**: Copied `/storage/emulated/0/Download/customers_q1.csv` to Telegram/Documents folder (file_events.csv).  
- **2026-02-15T01:07:58Z**: Sent Telegram message to @external_consult: "sending that sheet now" (app_db_messages.csv).  
- **2026-02-15T01:08:15Z**: Sent Telegram message to @external_consult: "delete after download" (app_db_messages.csv).  
- **2026-02-15T01:08:59Z**: Created `/storage/emulated/0/DCIM/Camera/IMG_20260215_010859.jpg` (file_events.csv).  
- **2026-02-15T01:09:44Z**: Sent Gmail message to "personal_mail": "see attached export" (app_db_messages.csv).  
- **2026-02-15T01:10:08Z**: Accessed Google Drive via Chrome (network_events.csv).  
- **2026-02-15T01:11:02Z**: Accessed internal corporate network `intranet.corp.local` (network_events.csv).  
- **2026-02-15T01:11:51Z**: Received Slack message in `#sales-ops`: "Need customer list for Monday forecast" (app_db_messages.csv).  
- **2026-02-15T01:12:04Z**: Deleted `/storage/emulated/0/Download/customers_q1.csv` (file_events.csv).  

**Note**: The Gmail message timestamp in the timeline (01:09:39Z) was corrected to match the explicit mapping (01:09:44Z).  

---

### **(2) Primary Hypothesis**  
**Data exfiltration via Telegram and Gmail** (confidence: 0.75).  
- The CSV file was created, copied to Telegram, and deleted after being shared (file_events.csv).  
- Telegram messages explicitly reference the file (e.g., "sending that sheet now," "delete after download") (app_db_messages.csv).  
- A Gmail message references an attachment likely tied to the CSV (app_db_messages.csv).  
- Network activity aligns with Telegram, Gmail, and Google Drive (network_events.csv).  

**Confidence Note**: Lowered from 0.85 to 0.75 due to gaps in file integrity verification and overclaiming about the Gmail attachment.  

---

### **(3) Alternative Hypotheses**  
1. **Synthetic Test Dataset Scenario** (confidence: 0.5).  
   - The artifact_manifest.json notes this is a "synthetic classroom dataset for reflection-pattern demo," suggesting the CSV and JPG might be test files. However, this note has not been cross-validated with other sources or metadata.  
   - The JPG file’s creation time (01:08:59Z) is close to Telegram/Gmail actions, but there is no evidence linking it to the CSV exfiltration.  

2. **Internal Collaboration via Corporate Network** (confidence: 0.4).  
   - The corporate network access (`intranet.corp.local`) and Slack request for a customer list (app_db_messages.csv) suggest internal sharing. However, the CSV deletion and external messaging (Telegram/Gmail) contradict this.  
   - No evidence confirms the CSV file was uploaded to Google Drive for internal use.  

---

### **(4) Confidence Scores**  
- **Primary Hypothesis**: 0.75 (supported by messages, network activity, and file actions, but with unresolved gaps).  
- **Alternative Hypotheses**: 0.5 (synthetic dataset context) and 0.4 (ambiguous internal/external intent).  

---

### **(5) Explicit Evidence Mapping**  
| Claim | Artifact | Evidence | Provenance Note |  
|-------|----------|---------|----------------|  
| CSV creation | file_events.csv | 2026-02-15T01:05:22Z `/storage/emulated/0/Download/customers_q1.csv` created | *File_events.csv was extracted from the acquisition but lacks hash validation.* |  
| CSV modification | file_events.csv | 2026-02-15T01:06:12Z `/storage/emulated/0/Download/customers_q1.csv` modified | *File_events.csv was extracted but no checksums verified.* |  
| CSV copied to Telegram | file_events.csv | 2026-02-15T01:07:41Z Telegram/Documents copy | *File_events.csv extracted; no validation of timestamp accuracy.* |  
| Telegram "sending that sheet" | app_db_messages.csv | 2026-02-15T01:07:58Z Telegram message | *Message log confirmed; no confirmation of sender/receiver.* |  
| Telegram "delete after download" | app_db_messages.csv | 2026-02-15T01:08:15Z Telegram message | *Message log confirmed; no recipient verification.* |  
| Gmail attachment reference | app_db_messages.csv | 2026-02-15T01:09:44Z Gmail message | *Message log confirms text, but no attachment file exists in dataset.* |  
| Google Drive access | network_events.csv | 2026-02-15T01:10:08Z drive.google.com | *Network activity recorded, but no file upload confirmed.* |  
| Corporate network access | network_events.csv | 2026-02-15T01:11:02Z intranet.corp.local | *Network activity recorded; no file transfer confirmed.* |  
| Slack customer list request | app_db_messages.csv | 2026-02-15T01:11:51Z Slack message | *Message log confirmed; no temporal link to CSV deletion.* |  
| CSV deletion | file_events.csv | 2026-02-15T01:12:04Z `/storage/emulated/0/Download/customers_q1.csv` deleted | *File_events.csv extracted; no timestamp validation.* |  

---

### **(6) Chain-of-Custody & Provenance**  
- **Acquisition**: The device was acquired via Cellebrite UFED 7.64 (artifact_manifest.json) and preserved as a read-only image (chain_of_custody.csv).  
- **File Integrity**: `file_events.csv` and `app_db_messages.csv` were extracted but not validated with checksums or hash comparisons to the original image.  
- **Timezone**: The device’s original timezone was UTC (artifact_manifest.json), and timestamps were adjusted to New York time for analysis. This assumes the device was in New York, but no timezone logs are available in the dataset.  
- **JPG File**: The creation of `/storage/emulated/0/DCIM/Camera/IMG_20260215_010859.jpg` (01:08:59Z) is timestamped closely to Telegram/Gmail activity but lacks contextual evidence of its relevance.  

---

### **(7) Recommendations**  
1. **Validate File Integrity**: Perform hash comparisons between `file_events.csv`, `app_db_messages.csv`, and the original acquisition image to verify authenticity.  
2. **Clarify Timestamps**: Confirm the Gmail message timestamp (01:09:44Z) is consistent across all logs and resolve discrepancies.  
3. **Avoid Speculative Links**: Clarify that the Gmail message mentions an attachment but does not confirm its existence in the dataset. Similarly, the Slack message is temporally close to CSV deletion but lacks causal evidence.  
4. **Document Custody**: Include chain-of-custody details (e.g., device handling, acquisition tool, preservation steps) to strengthen the evidentiary chain.  
5. **Contextualize Network Activity**: Explicitly link Google Drive access (01:10:08Z) and Gmail (01:09:44Z) to exfiltration or internal sharing only if file metadata or network traffic confirms the connection.  

---

### **Final Verdict**  
**Primary Hypothesis**: Plausible but requires further validation of file integrity and timestamp consistency.  
**Alternative Hypotheses**: Cannot be confirmed due to lack of cross-validated evidence.  
**Confidence**: Primary hypothesis at 0.75 (with caveats); alternative hypotheses at 0.5 and 0.4 respectively.  

**Note**: The report now addresses provenance gaps, clarifies timestamp inconsistencies, and avoids overclaiming. The primary hypothesis remains a working hypothesis but is not definitively proven without additional validation.

### And the iteration starts again ...

After **Generation Step (II)**, the corrected report is sent back to the reviewer.

The loop continues until the critique is `[[OK]]` or a max iteration limit is reached.

## Part 2: Run the Same Workflow with `ReflectionAgent`

Part I built the reflection workflow by hand: we created the generation history, produced a draft report, asked a reviewer model to critique it, and then fed that critique back into the next generation step.

Part II runs the same Lab 1 case through the packaged `ReflectionAgent` in the local `agentic_patterns` library. The goal is not to create a second lab, but to show how the same manual loop can be wrapped into a reusable class.

In [ ]:
import os
import sys
import inspect
from pathlib import Path

# Use the local standalone package copy whether the notebook is run from the repo root
# or from inside the lab folder.
src_dir_candidates = [Path("src"), Path("../src")]
for candidate in src_dir_candidates:
    if (candidate / "agentic_patterns").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise FileNotFoundError("Could not find local src/agentic_patterns folder")

from agentic_patterns import ReflectionAgent

sig = inspect.signature(ReflectionAgent.__init__)
if "client" not in sig.parameters:
    raise RuntimeError(
        f"Loaded ReflectionAgent without 'client' arg: {sig}. Restart kernel and run from this cell again."
    )

print("ReflectionAgent.__init__ signature:", sig)


In [ ]:
agent = ReflectionAgent(model=MODEL, client=client)


In [ ]:
generation_system_prompt = (
    "You are a digital forensics analyst tasked with writing a defensible incident report. "
    "Use only user-provided evidence and revise when critique is provided."
)

reflection_system_prompt = (
    "You are a senior digital forensics reviewer tasked with critique and recommendations. "
    "Output exactly [[OK]] when there is nothing material to improve."
)

user_msg = (
    "Analyze this mobile forensic case and produce a preliminary incident report.\n"
    "Return: (1) timeline, (2) primary hypothesis, (3) two alternative hypotheses, "
    "(4) confidence score 0-1 per claim, (5) explicit evidence mapping.\n\n"
    f"Artifacts:\n{artifact_bundle}"
)


In [ ]:
final_response = agent.run(
    user_msg=user_msg,
    generation_system_prompt=generation_system_prompt,
    reflection_system_prompt=reflection_system_prompt,
    n_steps=10,
    verbose=1,
)

### Final result

In [ ]:
display_markdown(final_response, raw=True)

### How this maps to the manual loop

- In Part I, you manually built the `generation_chat_history`; here, `ReflectionAgent.run(...)` creates and updates that generator context for you.
- In Part I, you manually built the `reflection_chat_history` and sent the draft for critique; here, the agent passes each draft into the reflection history automatically.
- In Part I, you manually fed critique back into the next generation step; here, the agent repeats that revise-and-review loop until `[[OK]]` appears or `n_steps` is reached.

The forensic task stays the same in both parts of the notebook. What changes is the level of abstraction: first you see the loop directly, then you reuse it through a local class.
